In [1]:
import torch.nn as nn
import torchvision
import torch
from torchinfo import summary
from DWT import DWT_2D
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import random
import pandas as pd
import ast
from scipy import stats
from tqdm import tqdm

import os
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score, precision_recall_fscore_support
from sklearn.metrics import roc_curve, auc, accuracy_score, confusion_matrix, f1_score, precision_recall_curve, precision_score, recall_score, cohen_kappa_score
from sklearn.utils import resample
import seaborn as sns
from scipy.stats import pearsonr

from IXCM import IXCNet

In [2]:
class TB11KDataset(Dataset):
    def __init__(self, list_path, img_root_path, transform=None):
        """
        Args:
            list_path: txt索引文件路径 (e.g., ./TB11K/lists/TBX11K_train.txt)
            img_root_path: 图片根目录 (e.g., ./TB11K/img)
            transform: 图片预处理
        """
        self.img_root_path = img_root_path
        self.transform = transform
        self.img_paths = []
        self.labels = []

        with open(list_path, 'r') as f:
            lines = f.readlines()
            for line in lines:
                # 假设txt格式为: "folder/filename.png label" 或 纯路径
                # 这里我们只取第一部分的路径字符串
                parts = line.strip().split()
                rel_path = parts[0]
                
                # 路径修正：TB11K txt中有时包含 'imgs/' 前缀，需根据实际情况调整
                # 你的描述是图片在 ./TB11K/img/health 等，所以我们拼接路径
                full_path = os.path.join(self.img_root_path, rel_path)
                
                # 标签逻辑: health/sick -> 0 (Non-TB), tb -> 1 (TB)
                # 检查路径字符串中是否包含目录名
                # 注意：根据你的描述，sick是患病非结核，health是健康，都算0
                # tb是结核，算1
                # 路径通常包含 'health', 'sick', 'tb' 字符串
                if '/tb/' in rel_path or rel_path.startswith('tb/'):
                    label = 1.0
                else:
                    label = 0.0
                
                self.img_paths.append(full_path)
                self.labels.append(label)

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        label = self.labels[idx]

        # 打开图片并转为灰度 (你的DenseNet修改了conv0接受1通道)
        try:
            image = Image.open(img_path).convert('L') 
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            # 返回一个黑图防止崩溃，或者你可以选择跳过
            image = Image.new('L', (256, 256))

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor([label], dtype=torch.float32)

In [3]:
def seed_everything(seed=42):
    """
    固定所有随机种子以保证实验可复现性
    """
    # 1. Python标准库
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. NumPy
    np.random.seed(seed)
    
    # 3. PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # 如果使用多卡
    
    # 4. CuDNN (最关键的一步)
    # 保证卷积算法选择的确切性
    torch.backends.cudnn.deterministic = True 
    # 禁止寻找最优卷积算法（虽然会稍微降低速度，但能保证完全一致）
    torch.backends.cudnn.benchmark = False

    print(f"Random Seed set to: {seed}")

In [4]:
# ==========================================
# 辅助函数：只负责跑推理拿数据
# ==========================================
def get_preds_and_labels(model, dataloader, device, img_dec=True):
    """
    运行推理，返回:
    1. y_scores: 主分支融合概率
    2. y_s_scores: 结构分支概率 (alpha_s / (alpha_s + beta_s))
    3. y_a_scores: 纹理分支概率 (alpha_a / (alpha_a + beta_a))
    4. y_true: 真实标签
    """
    model.eval()
    all_preds = []      # 存主输出 y
    all_preds_s = []    # 存结构分支 y_s
    all_preds_a = []    # 存纹理分支 y_a
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc="Running Inference"):
            inputs = inputs.to(device)
            labels = labels.to(device).float()
            
            # 1. 获取模型输出
            # alphas, betas 形状: [Batch, 2 * Num_Classes]
            # outs 形状: [Batch, Num_Classes] (经过 Sigmoid)
            alphas, betas, outs = model(inputs)
            if img_dec:
                # 2. 拆分 S 和 A 的参数
                # [Batch, Num_Classes]
                alpha_a = alphas[:, 1::2] 
                beta_a  = betas[:, 1::2]
                
                alpha_s = alphas[:, 0::2]
                beta_s  = betas[:, 0::2]
                
                # 3. 计算分支概率 (Expectation = alpha / (alpha + beta))
                # 加 1e-8 防止除零 (虽然 alpha>=1)
                prob_s = alpha_s / (alpha_s + beta_s + 1e-8)
                prob_a = alpha_a / (alpha_a + beta_a + 1e-8)

                all_preds_s.append(prob_s.cpu().numpy())
                all_preds_a.append(prob_a.cpu().numpy())
            else:
                alpha_s = alphas
                beta_s  = betas
                prob_s = alpha_s / (alpha_s + beta_s + 1e-8)
                all_preds_s.append(prob_s.cpu().numpy())
            
            # 4. 收集结果
            all_preds.append(outs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            
    # 拼接
    y_scores = np.concatenate(all_preds, axis=0)
    y_s_scores = np.concatenate(all_preds_s, axis=0)
    if img_dec:
        y_a_scores = np.concatenate(all_preds_a, axis=0)
    else:
        y_a_scores = np.concatenate([[0]], axis=0)
    y_true = np.concatenate(all_labels, axis=0)
    
    return y_scores, y_s_scores, y_a_scores, y_true

# ==========================================
# 在验证集上寻找最优阈值
# ==========================================
def find_optimal_thresholds(model, val_loader, device, label_columns, mode='youden'):
    """
    遍历验证集，为每个类别寻找最佳阈值。
    
    Args:
        mode (str): 'f1' - 最大化 F1 Score (偏向 Precision/Recall 的调和平均)
                    'youden' - 最大化约登指数 (Sensitivity + Specificity - 1)
    """
    assert mode in ['f1', 'youden', 'distance', 'mid'], "Mode must be 'f1' or 'youden' or 'distance'"
    
    print(f"{'='*20} Finding Optimal Thresholds on Val Set (Mode: {mode}) {'='*20}")
    
    # 1. 获取验证集的预测结果 (假设 get_preds_and_labels 已经在外部定义好)
    y_scores, _, _, y_true = get_preds_and_labels(model, val_loader, device)
    
    optimal_thresholds = []
    best_scores = []  # 记录最佳的 F1 或 J 值
    
    print("Searching best thresholds for each class...")
    for i, class_name in enumerate(label_columns):
        cls_true = y_true[:, i]
        cls_score = y_scores[:, i]
        
        # --- 异常处理：如果没有正样本或负样本 ---
        if np.sum(cls_true) == 0 or np.sum(cls_true) == len(cls_true):
            print(f"  [Warning] Class {class_name} has no positive/negative samples in Val Set. Using default 0.5")
            optimal_thresholds.append(0.5)
            best_scores.append(0.0)
            continue

        best_threshold = 0.5
        best_score = 0.0

        # ==================== 模式 A: F1-Score ====================
        if mode == 'f1':
            # precision_recall_curve 返回的 P/R 长度比 thresholds 多 1
            precision, recall, thresholds = precision_recall_curve(cls_true, cls_score)
            
            # 计算 F1
            numerator = 2 * recall * precision
            denominator = recall + precision + 1e-8
            f1_scores = np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator!=0)
            
            # 找到最大 F1 的索引
            best_idx = np.argmax(f1_scores)
            
            # 处理索引越界 (thresholds 比 f1 短 1)
            # 如果最佳点是最后一个点（Recall=0, Precision=1），通常意味着阈值设得极高
            if best_idx < len(thresholds):
                best_threshold = thresholds[best_idx]
            else:
                best_threshold = 0.5 # 兜底
            
            best_score = f1_scores[best_idx]

        # ==================== 模式 B: Youden's Index ====================
        elif mode == 'youden':
            # roc_curve 返回的 fpr, tpr, thresholds 长度是一致的
            fpr, tpr, thresholds = roc_curve(cls_true, cls_score)
            
            # 计算约登指数 J = Sensitivity + Specificity - 1
            # 因为 Specificity = 1 - FPR
            # 所以 J = TPR + (1 - FPR) - 1 = TPR - FPR
            j_scores = tpr - fpr
            
            # 找到最大 J 的索引
            best_idx = np.argmax(j_scores)
            
            best_threshold = thresholds[best_idx]
            best_score = j_scores[best_idx]
            
            # 注意：roc_curve 的 thresholds[0] 通常是 max(score) + 1 (用于表示全负预测)
            # 如果选到了这个点，说明模型倾向于把所有样本判负，需要兜底限制一下范围
            if best_threshold > 1.0:
                 best_threshold = 1.0
        elif mode == 'distance':
            fpr, tpr, thresholds = roc_curve(cls_true, cls_score)

            dists = np.sqrt((1 - tpr)**2 + fpr**2)

            best_idx = np.argmin(dists)
            best_threshold = thresholds[best_idx]
            best_score = dists[best_idx]
        elif mode == 'mid':
            best_threshold = 0.5

        # ==================== 记录结果 ====================
        optimal_thresholds.append(best_threshold)
        best_scores.append(best_score)
        
        print(f"  Class: {class_name:<20} | Best Threshold: {best_threshold:.4f} | Val {mode}: {best_score:.4f}")
        
    return np.array(optimal_thresholds)
    
def calculate_consistency_bootstrap(scores_main, thresh_main, scores_sub, thresh_sub, 
                                    n_bootstraps=1000, alpha=0.95, seed=42):
    """
    计算两个分支预测结果的一致性 (Cohen's Kappa) 并估计 95% CI。
    支持非对称阈值：主分支用 Optimal Threshold，子分支用 Fixed 0.5。
    
    Args:
        scores_main: 主分支概率值
        thresh_main: 主分支阈值
        scores_sub:  子分支概率值 (y_s 或 y_a)
        thresh_sub:  子分支阈值 (通常为 0.5)
    
    Returns:
        tuple: (original_kappa, ci_lower, ci_upper)
    """
    rng = np.random.RandomState(seed)
    n_samples = len(scores_main)
    
    # 1. 计算原始 Kappa (Point Estimate)
    bin_main = (scores_main > thresh_main).astype(int)
    bin_sub = (scores_sub > thresh_sub).astype(int)
    
    try:
        kappa_orig = cohen_kappa_score(bin_main, bin_sub)
    except:
        kappa_orig = 0.0
        
    boot_kappas = []
    
    # 2. Bootstrap 循环
    for i in range(n_bootstraps):
        # 重采样索引
        indices = rng.randint(0, n_samples, n_samples)
        
        s_main = scores_main[indices]
        s_sub = scores_sub[indices]
        
        # 应用各自的阈值进行二值化
        b_main = (s_main > thresh_main).astype(int)
        b_sub = (s_sub > thresh_sub).astype(int)
        
        try:
            k = cohen_kappa_score(b_main, b_sub)
            if not np.isnan(k):
                boot_kappas.append(k)
        except:
            pass
            
    # 3. 计算置信区间
    boot_kappas = np.array(boot_kappas)
    if len(boot_kappas) == 0:
        return kappa_orig, 0.0, 0.0
        
    boot_kappas.sort()
    
    lower_idx = int((1.0 - alpha) / 2.0 * len(boot_kappas))
    upper_idx = int((1.0 - (1.0 - alpha) / 2.0) * len(boot_kappas))
    
    # 索引边界保护
    lower_idx = max(0, lower_idx)
    upper_idx = min(len(boot_kappas) - 1, upper_idx)
    
    return kappa_orig, boot_kappas[lower_idx], boot_kappas[upper_idx]
# ==============================================================================
# 1. 核心工具函数：Bootstrap CI 计算器
# ==============================================================================
def calculate_bootstrap_batch(y_true, y_score, threshold, binary_metrics, prob_metrics, 
                              n_bootstraps=1000, alpha=0.95, seed=42):
    """
    一次性对所有指标进行 Bootstrap CI 计算，避免重复采样。
    
    Args:
        y_true: 真实标签
        y_score: 预测概率/Logits
        threshold: 当前类别的最佳阈值 (用于生成二值预测)
        binary_metrics: 字典 {指标名: 计算函数}，接收 (y_true, y_pred_bin)
        prob_metrics: 字典 {指标名: 计算函数}，接收 (y_true, y_score)
    
    Returns:
        results: 字典 {指标名: (original_score, ci_lower, ci_upper)}
    """
    rng = np.random.RandomState(seed)
    
    # 1. 在原始数据上计算点估计值 (Original Scores)
    # ---------------------------------------------------
    y_pred_bin_orig = (y_score > threshold).astype(int)
    results = {}
    
    # 计算二值指标原始值
    for name, func in binary_metrics.items():
        try:
            results[name] = {'original': func(y_true, y_pred_bin_orig), 'bootstraps': []}
        except:
            results[name] = {'original': 0.0, 'bootstraps': []}

    # 计算概率指标原始值 (如 AUC)
    for name, func in prob_metrics.items():
        try:
            results[name] = {'original': func(y_true, y_score), 'bootstraps': []}
        except:
            results[name] = {'original': 0.0, 'bootstraps': []}

    # 2. Bootstrap 循环 (只循环 1000 次，内部计算所有指标)
    # ---------------------------------------------------
    n_samples = len(y_true)
    for i in range(n_bootstraps):
        # 生成一次索引，复用于所有指标
        indices = rng.randint(0, n_samples, n_samples)
        
        y_true_boot = y_true[indices]
        y_score_boot = y_score[indices]
        
        # 边界检查：如果采样后只有一类，跳过这次循环
        if len(np.unique(y_true_boot)) < 2:
            continue
            
        # 生成当前采样的二值预测
        y_pred_bin_boot = (y_score_boot > threshold).astype(int)
        
        # 批量计算二值指标
        for name, func in binary_metrics.items():
            try:
                score = func(y_true_boot, y_pred_bin_boot)
                results[name]['bootstraps'].append(score)
            except ValueError:
                pass
        
        # 批量计算概率指标
        for name, func in prob_metrics.items():
            try:
                score = func(y_true_boot, y_score_boot)
                results[name]['bootstraps'].append(score)
            except ValueError:
                pass

    # 3. 计算置信区间 CI
    # ---------------------------------------------------
    final_output = {}
    
    for name, data in results.items():
        original = data['original']
        scores = np.array(data['bootstraps'])
        
        if len(scores) == 0:
            final_output[name] = (original, 0.0, 0.0)
            continue
            
        scores.sort()
        
        # 计算分位点索引
        lower_idx = int((1.0 - alpha) / 2.0 * len(scores))
        upper_idx = int((1.0 - (1.0 - alpha) / 2.0) * len(scores))
        
        # 索引保护
        lower_idx = max(0, lower_idx)
        upper_idx = min(len(scores) - 1, upper_idx)
        
        ci_lower = scores[lower_idx]
        ci_upper = scores[upper_idx]
        
        final_output[name] = (original, ci_lower, ci_upper)
        
    return final_output

# ==============================================================================
# 2. 辅助函数：自定义指标计算逻辑
# ==============================================================================
def calc_specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return tn / (tn + fp + 1e-8)

def calc_npv(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return tn / (tn + fn + 1e-8)

def calc_sensitivity(y_true, y_pred):
    return recall_score(y_true, y_pred, zero_division=0)

def calc_precision(y_true, y_pred):
    return precision_score(y_true, y_pred, zero_division=0)

def calc_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, zero_division=0)

def calc_acc(y_true, y_pred):
    return accuracy_score(y_true, y_pred)

def calc_auc(y_true, y_score):
    return roc_auc_score(y_true, y_score)

# ==============================================================================
# 3. 主评估函数 (使用优化后的 Batch Bootstrap)
# ==============================================================================

def evaluate_model_comprehensive(model, dataloader, device, label_columns, save_dir, thresholds=None, img_dec=True, evidence=True):
    model.eval()
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    print(f"\n{'='*20} Final Evaluation & Consistency Analysis {'='*20}")
    
    # 1. 获取所有分支预测值
    y_scores, y_s_scores, y_a_scores, y_true = get_preds_and_labels(model, dataloader, device, img_dec=img_dec)
    
    metrics_data = []
    consistency_data = [] 
    roc_curve_data = {} 
    
    # 2. 阈值设定
    if thresholds is None:
        print("Using default 0.5 threshold for Main Branch.")
        main_thresholds = np.full(len(label_columns), 0.5)
    else:
        print("Using provided optimal thresholds for Main Branch.")
        main_thresholds = thresholds
        
    fixed_thresholds = np.full(len(label_columns), 0.5)
    
    # ========================== Part A: 基础指标评估 ==========================
    if evidence:
        if img_dec:
            eval_configs = [
                ('Main', y_scores, main_thresholds),
                ('Structure', y_s_scores, fixed_thresholds),
                ('Appearance', y_a_scores, fixed_thresholds)
            ]
        else:
            eval_configs = [
                ('Main', y_scores, main_thresholds),
                ('Structure', y_s_scores, fixed_thresholds)
            ]
    else:
        eval_configs = [
            ('Main', y_scores, main_thresholds)
        ]

    plt.figure(figsize=(12, 10))
    colors = sns.color_palette("husl", len(label_columns))
    
    binary_funcs = {
        'Accuracy': calc_acc, 'Sensitivity': calc_sensitivity, 'Specificity': calc_specificity,
        'Precision': calc_precision, 'NPV': calc_npv, 'F1_Score': calc_f1
    }
    prob_funcs = {'AUC': calc_auc}

    for branch_name, scores, current_thresholds_list in eval_configs:
        print(f"\n>>> Evaluating Branch: {branch_name} ...")
        for i, class_name in enumerate(label_columns):
            cls_true = y_true[:, i]
            cls_score = scores[:, i]
            current_threshold = current_thresholds_list[i]
            
            row_data = {'Branch': branch_name, 'Class': class_name, 'Threshold': current_threshold}
            
            # Bootstrap 计算基础指标
            batch_results = calculate_bootstrap_batch(
                cls_true, cls_score, current_threshold, binary_funcs, prob_funcs, n_bootstraps=1000
            )
            for metric_name, (score, lower, upper) in batch_results.items():
                row_data[metric_name] = score
                row_data[f'{metric_name}_CI_Lower'] = lower
                row_data[f'{metric_name}_CI_Upper'] = upper

            # 混淆矩阵统计
            cls_pred_bin = (cls_score > current_threshold).astype(int)
            tn, fp, fn, tp = confusion_matrix(cls_true, cls_pred_bin, labels=[0, 1]).ravel()
            row_data.update({'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn})

            # 仅绘制 Main 的 ROC
            if branch_name == 'Main':
                fpr, tpr, roc_thresholds = roc_curve(cls_true, cls_score)
                roc_curve_data[class_name] = {'fpr': fpr, 'tpr': tpr, 'auc': row_data['AUC']}
                plt.plot(fpr, tpr, color=colors[i], lw=2, label=f'{class_name} (AUC={row_data["AUC"]:.4f})')

            metrics_data.append(row_data)

    # ========================== Part B: 一致性分析 (Kappa with Bootstrap) ==========================
    if evidence:
        print(f"\n>>> Calculating Consistency Metrics (Kappa & Pearson) with Bootstrap ...")
        
        for i, class_name in enumerate(label_columns):
            prob_main = y_scores[:, i]
            prob_s = y_s_scores[:, i]
            if img_dec:
                prob_a = y_a_scores[:, i]
            
            thresh_main = main_thresholds[i]
            thresh_sub = 0.5  # S和A固定用0.5
            
            # --- 1. S vs Main ---
            kappa_s, k_s_low, k_s_high = calculate_consistency_bootstrap(
                prob_main, thresh_main, prob_s, thresh_sub, n_bootstraps=1000
            )
            corr_s, _ = pearsonr(prob_main, prob_s)
            
            consistency_data.append({
                'Class': class_name,
                'Comparison': 'S_vs_Main',
                'Kappa': kappa_s,
                'Kappa_CI_Lower': k_s_low,
                'Kappa_CI_Upper': k_s_high,
                'Pearson_Corr': corr_s
            })
            if img_dec:
                # --- 2. A vs Main ---
                kappa_a, k_a_low, k_a_high = calculate_consistency_bootstrap(
                    prob_main, thresh_main, prob_a, thresh_sub, n_bootstraps=1000
                )
                corr_a, _ = pearsonr(prob_main, prob_a)
                
                consistency_data.append({
                    'Class': class_name,
                    'Comparison': 'A_vs_Main',
                    'Kappa': kappa_a,
                    'Kappa_CI_Lower': k_a_low,
                    'Kappa_CI_Upper': k_a_high,
                    'Pearson_Corr': corr_a
                })
            if img_dec:
                print(f"  [{class_name}] Kappa(S,Main)={kappa_s:.3f} | Kappa(A,Main)={kappa_a:.3f}")
            else:
                print(f"  [{class_name}] Kappa(S,Main)={kappa_s:.3f}")

    # ========================== Part C: 保存结果 ==========================
    
    # 1. 保存 ROC 图
    plt.plot([0, 1], [0, 1], 'k--', lw=2, alpha=0.5)
    plt.title('Main Branch ROC Curves'); plt.legend(loc="lower right", fontsize=8)
    plt.savefig(os.path.join(save_dir, 'main_roc.png'), dpi=300); plt.close()
    
    # 2. 保存基础指标 CSV
    df_metrics = pd.DataFrame(metrics_data)
    # 调整列顺序，确保分支顺序
    branch_order = pd.CategoricalDtype(['Main', 'Structure', 'Appearance'], ordered=True)
    df_metrics['Branch'] = df_metrics['Branch'].astype(branch_order)
    df_metrics = df_metrics.sort_values(by=['Branch', 'Class'])
    
    # 计算 Macro Avg
    numeric_cols = df_metrics.select_dtypes(include=[np.number]).columns
    macro_avg = df_metrics.groupby('Branch', observed=True)[numeric_cols].mean().reset_index()
    macro_avg['Class'] = 'Macro Average'
    df_final = pd.concat([df_metrics, macro_avg], ignore_index=True)
    
    df_final.to_csv(os.path.join(save_dir, 'final_metrics.csv'), index=False)
    if evidence:
        # 3. 保存一致性分析 CSV
        df_consistency = pd.DataFrame(consistency_data)
        cols = ['Class', 'Comparison', 'Kappa', 'Kappa_CI_Lower', 'Kappa_CI_Upper', 'Pearson_Corr']
        df_consistency = df_consistency[cols]
        df_consistency.to_csv(os.path.join(save_dir, 'consistency_analysis.csv'), index=False)
        
        print(f"Evaluation Complete. Files saved to {save_dir}")
    else:
        df_consistency = pd.DataFrame(consistency_data)
    return df_final, df_consistency

In [5]:
# --- 配置参数 ---
evidence=False
img_dec=False
seed_everything(42) # 固定随机种子保证结果可复现
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if img_dec:
    BATCH_SIZE = 64
else:
    BATCH_SIZE = 32
# 路径配置
IMG_ROOT = '/teams/Xray_1751630141/TB11K/imgs'
TRAIN_LIST = '/teams/Xray_1751630141/TB11K/TBX11K/lists/TBX11K_train.txt'
VAL_LIST = '/teams/Xray_1751630141/TB11K/TBX11K/lists/TBX11K_val.txt'
# --- 数据准备 ---
# 你的DenseNet conv0改为了1通道，且 IXCNet 需要 DWT，建议尺寸保持 256 或 512
# DWT 通常要求输入尺寸是偶数
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        # 如果是灰度图，ToTensor后是 [1, H, W]，通常不需要 Normalize 或者用 0.5
        transforms.Normalize([0.5], [0.5]) 
    ]),
    'val': transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])
    ]),
}

train_dataset = TB11KDataset(TRAIN_LIST, IMG_ROOT, transform=data_transforms['train'])
val_dataset = TB11KDataset(VAL_LIST, IMG_ROOT, transform=data_transforms['val'])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

Random Seed set to: 42
Train samples: 6600, Val samples: 1800


In [6]:
# ==========================================
# 确保 eval_output_dir 定义在主脚本里
result_root_path = f'./result/TB11K/0131_wd0_0001_gc1_0_vw1_0_lm_{'bcevar' if evidence else 'bce'}_lr_0.0001_kllmd_0.1_imgdec_{int(img_dec)}'
val_eval_output_dir = os.path.join(result_root_path, 'val_evaluation_results')
train_eval_output_dir = os.path.join(result_root_path, 'train_evaluation_results')

# 加载最佳模型
final_model = IXCNet(evi_vec_dim=64, wavename='haar', ydim=1, in_channels=1, img_dec=img_dec).to(DEVICE)
final_model.to(DEVICE)
best_model_path = os.path.join(result_root_path, 'best_model_auc.pth')
if os.path.exists(best_model_path):
    print(f"Loading weights from: {best_model_path}")
    final_model.load_state_dict(torch.load(best_model_path))
else:
    print("Warning: Best model not found, using current weights.")

LABEL_COLUMNS = ['TB']
optimal_thresholds = find_optimal_thresholds(
    final_model, 
    val_loader,
    DEVICE, 
    LABEL_COLUMNS,
    mode='mid'
)
# 运行
df_val_results, df_val_consist = evaluate_model_comprehensive(
    final_model, 
    val_loader, 
    DEVICE, 
    LABEL_COLUMNS, 
    val_eval_output_dir,
    thresholds=optimal_thresholds,
    img_dec=img_dec,
    evidence=evidence
)
df_train_results, df_train_consist = evaluate_model_comprehensive(
    final_model, 
    train_loader, 
    DEVICE, 
    LABEL_COLUMNS, 
    train_eval_output_dir,
    thresholds=optimal_thresholds,
    img_dec=img_dec,
    evidence=evidence
)

✅ Building IXCNet with backbone: densenet121


/root/miniconda3/envs/myconda/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/root/miniconda3/envs/myconda/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Loading weights from: ./result/TB11K/0131_wd0_0001_gc1_0_vw1_0_lm_bce_lr_0.0001_kllmd_0.1_imgdec_0/best_model_auc.pth
==================== Finding Optimal Thresholds on Val Set (Mode: mid) ====================


Running Inference: 100%|██████████| 57/57 [00:05<00:00,  9.58it/s]


Searching best thresholds for each class...
  Class: TB                   | Best Threshold: 0.5000 | Val mid: 0.0000

==================== Final Evaluation & Consistency Analysis ====================


Running Inference: 100%|██████████| 57/57 [00:05<00:00, 10.20it/s]


Using provided optimal thresholds for Main Branch.

>>> Evaluating Branch: Main ...

==================== Final Evaluation & Consistency Analysis ====================


Running Inference: 100%|██████████| 207/207 [00:20<00:00, 10.25it/s]


Using provided optimal thresholds for Main Branch.

>>> Evaluating Branch: Main ...
